In [39]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import sys 
sys.path.append("../")

C:\Users\Lenovo\Desktop\Diabetes_Classifier


In [ ]:
from modeling.XGBoost import XGBoost
from modeling.RandomForest import RandomForest
from modeling.MLP import MLP
from modeling.Ensemble import LR
from src.data_feature_engineering import features
from src.data_preprocessing import scaling
from src.configuration import CLEAN_DATA
import pandas as pd
import numpy as np
from modeling.Ada import Ada
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score,recall_score,precision_score,balanced_accuracy_score,roc_auc_score,accuracy_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:

df=pd.read_csv(CLEAN_DATA)
df.head()
df.columns = df.columns.str.strip()

In [43]:

target='diagnosis'
y = df[target]
X = df.drop(target, axis=1)
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2, random_state=42
)

X_train,X_test=features(X_train,X_test)

Spliting the data for the stacking ensemble!

In [45]:

xgb=XGBoost()
xgb_params={'n_estimators': 7800, 'max_depth': 13, 'learning_rate': 0.03585979413065538, 'gamma': 0.00025046646096832056, 'min_child_weight': 8, 'reg_alpha': 9.979812055651985, 'reg_lambda': 0.041477402642739976, 'subsample': 0.9999355054935987, 'colsample_bytree': 0.8896668801909641, 'tree_method': 'hist', 'n_jobs': -1}
xgb.set_params(xgb_params)
pd.set_option('display.max_rows',None)
m=xgb.evaluation(X_train,y_train)
m.out()
#Optuna
# params=xgb.hyperparameter_tuning(X,y,30)
# xgb.set_params(params)



Accuracy: 0.8211054335878314, std: 0.007552128983431863
Precision: 0.7906230752549798, std: 0.0064086325797492235
F1: 0.7582444562941303, std: 0.012745598418396556
Recall: 0.7286287585353193, std: 0.020525890375465608
Balanced Accuracy:0.8038456538236867, std: 0.009875240964343364
ROC-AUC: 0.909476103773206, std: 0.008395194278684586


In [46]:
rf=RandomForest()
rf_params={'n_estimators': 1200, 'max_depth': 6, 'min_samples_split': 18, 'min_samples_leaf': 9, 'max_features': 'log2', 'bootstrap': True, 'criterion': 'gini', 'class_weight': None, 'n_jobs': -1, 'random_state': 42}
# rf_params=rf.hyperparameter_tuning(X_train,y_train,30)
rf.set_params(rf_params)
m=rf.evaluation(X_train,y_train)
m.out()



Accuracy: 0.826223819850866, std: 0.005671249386990618
Precision: 0.8105066485488361, std: 0.013904881000000797
F1: 0.7606712180060415, std: 0.010140733715644144
Recall: 0.7172403466038414, std: 0.022540498355764212
Balanced Accuracy:0.8058808993745279, std: 0.007690306736198796
ROC-AUC: 0.9120667249909914, std: 0.007793769720455349


In [47]:

pd.set_option('display.max_columns',None)
X_train.head(5)

,age,bmi,chol,tg,hdl,ldl,cr,bun,lipids,Age_x_BMI,HDL_x_LDL,BMI/LDL,BMI/HDL+LDL,bun_x_cr,chol/ldl,PCA_1,PCA_2,PCA_3,PCA_4,PCA_5,PCA_6,PCA_7,cluster_labels,Cluster_0,Cluster_1,Cluster_2,Cluster_3,Cluster_4,Cluster_5,gender
1215,54,25,5.28,3.39,1.08,2.89,81.7,4.65,0.373702,1350,3.1212,8.650519,6.297229,379.905,1.826990,-1.365042,-0.712553,0.199970,1.355497,0.522417,0.644990,0.108773,0,1.779536,13.501025,2.926794,43.280315,12.683027,2.725953,1
19,27,20,5.86,2.71,1.80,2.94,51.1,3.10,0.612245,540,5.2920,6.802721,4.219409,158.410,1.993197,0.404395,0.307139,-1.738320,-0.046443,2.021914,0.902333,-0.117368,5,3.573916,11.990953,4.966122,45.147112,12.917012,2.899893,0
2093,35,17,4.35,0.60,1.48,2.27,56.0,3.43,0.651982,595,3.3596,7.488987,4.533333,192.080,1.916300,-0.708258,-0.262957,-1.643508,-1.966113,1.113553,0.310511,0.114481,5,3.637299,13.113079,4.545817,44.859405,13.073248,1.691986,0
668,41,18,4.00,1.00,1.00,2.00,51.0,3.00,0.500000,738,2.0000,9.000000,6.000000,153.000,2.000000,-2.084440,-0.344952,-1.960962,-1.347430,0.632690,0.391092,0.095615,5,3.875563,14.377875,3.876160,45.121231,12.602677,2.086863,0
218,49,25,4.20,1.10,1.10,2.70,53.0,4.30,0.407407,1225,2.9700,9.259259,6.578947,227.900,1.555556,-1.143659,-1.251269,-1.160571,-0.221287,-0.786629,-0.198333,0.203089,5,2.278692,13.421593,3.312138,44.451823,13.500593,1.739787,0


In [48]:
X_train_scaled,X_test_scaled=scaling(X_train,X_test)
X_eval_scaled=pd.concat([X_train_scaled,X_test_scaled])
y_eval_scaled=y_eval
X_train_scaled.head()

,age,bmi,chol,tg,hdl,ldl,cr,bun,lipids,Age_x_BMI,HDL_x_LDL,BMI/LDL,BMI/HDL+LDL,bun_x_cr,chol/ldl,PCA_1,PCA_2,PCA_3,PCA_4,PCA_5,PCA_6,PCA_7,cluster_labels,Cluster_0,Cluster_1,Cluster_2,Cluster_3,Cluster_4,Cluster_5,gender_1
0,0.367086,0.106233,0.435280,1.232226,-0.496700,-0.020071,0.508351,-0.137113,-0.586245,0.301378,-0.369708,-0.179258,0.100652,0.051529,0.051101,-0.359751,-0.293910,0.090380,0.988486,0.448502,0.751809,0.157244,-1.052904,-0.614439,0.236851,-0.552577,-0.264939,-0.272359,-0.346686,1.0
1,-1.562416,-1.066098,1.030999,0.727563,0.194750,0.033323,-0.892544,-1.054058,0.135381,-1.519612,0.020802,-0.568270,-0.815401,-0.580954,0.317147,0.106577,0.126687,-0.785668,-0.033868,1.735842,1.051772,-0.169668,1.216515,-0.159416,-0.162589,-0.022118,0.674167,-0.193168,-0.303517,0.0
2,-0.990712,-1.769497,-0.519923,-0.838375,-0.112561,-0.682151,-0.668217,-0.858838,0.255592,-1.395965,-0.326821,-0.423792,-0.677001,-0.484808,0.194058,-0.186659,-0.108463,-0.742816,-1.433773,0.956001,0.361936,0.165495,1.216515,-0.143343,0.134232,-0.131445,0.529434,-0.140292,-0.603302,0.0
3,-0.561933,-1.535031,-0.879408,-0.541515,-0.573528,-0.970477,-0.897122,-1.113216,-0.204176,-1.074481,-0.571403,-0.105682,-0.030388,-0.596402,0.328036,-0.549346,-0.142284,-0.886296,-0.982603,0.543174,0.455863,0.138221,1.216515,-0.082924,0.468792,-0.305633,0.661147,-0.299553,-0.505299,0.0
4,0.009771,0.106233,-0.673988,-0.467300,-0.477493,-0.222966,-0.805560,-0.344165,-0.484282,0.020361,-0.396907,-0.051101,0.224854,-0.382524,-0.383380,-0.301407,-0.516116,-0.524543,-0.161372,-0.675333,-0.231180,0.293587,1.216515,-0.487862,0.215840,-0.452344,0.324397,0.004340,-0.591438,0.0


In [49]:
mlp=MLP()
mlp_params={'hidden_layer_sizes': (64,), 'activation': 'relu', 'solver': 'adam', 'alpha': 6.120954197759074e-05, 'learning_rate_init': 0.006507372434299954, 'batch_size': 64, 'max_iter': 900, 'early_stopping': True, 'random_state': 42}
# params=mlp.hyperparameter_tuning(X_train_scaled,y_train,40)
# mlp.set_params(params)

mlp.set_params(mlp_params)



m=mlp.evaluation(X_train_scaled,y_train)
m.out()




Accuracy: 0.820132794628799, std: 0.006441649253414615
Precision: 0.8036655789699555, std: 0.03551955484142718
F1: 0.7521281377359734, std: 0.016292030621231905
Recall: 0.7108972567184443, std: 0.050295340729529264
Balanced Accuracy:0.7997363071270991, std: 0.011936317287299608
ROC-AUC: 0.9075422822329123, std: 0.008394618478146243


In [ ]:
ada = Ada()
ada_params={'n_estimators': 364, 'learning_rate': 0.6133741065916875}
# ada_params=ada.hyperparameter_tuning(X_train,y_train,20)
ada.set_params(ada_params)
m=ada.evaluation(X_train,y_train,5)
m.out()

Accuracy: 0.8254950833308576, std: 0.006743498394237216
Precision: 0.76954991830925, std: 0.013182172917704344
F1: 0.7752848154364826, std: 0.010428069404950547
Recall: 0.7817573773110251, std: 0.023707496643022973
Balanced Accuracy:0.8173265120092765, std: 0.008481058504740898
ROC-AUC: 0.9090322613202101, std: 0.00751175508581043


Cross Validating the Base models to see which model performs the best and also performing an Optuna Study to find best parameters!

In [52]:
ensemble=[rf,xgb,ada,mlp]
X_train_copy=X_train_scaled.copy()
X_test_copy=X_test_scaled.copy()
oof=pd.DataFrame()
oof_test=pd.DataFrame()
for model in ensemble:
    oof[f'OOF_{type(model).__name__}'],oof_test[f'OOF_{type(model).__name__}']=model.oof(X_train,y_train,X_test,type(model).__name__,5)

RandomForest fold 1/5 done
RandomForest fold 2/5 done
RandomForest fold 3/5 done
RandomForest fold 4/5 done
RandomForest fold 5/5 done
XGBoost fold 1/5 done
XGBoost fold 2/5 done
XGBoost fold 3/5 done
XGBoost fold 4/5 done
XGBoost fold 5/5 done
Ada fold 1/5 done
Ada fold 2/5 done
Ada fold 3/5 done
Ada fold 4/5 done
Ada fold 5/5 done
MLP fold 1/5 done
MLP fold 2/5 done
MLP fold 3/5 done
MLP fold 4/5 done
MLP fold 5/5 done


Out of Fold predictions from each Base model!

In [53]:
meta_x_model=LR()
params={'penalty': 'l2', 'l1_ratio': 0.377184927025366, 'C': 0.3926479245177274, 'class_weight': 'balanced','n_jobs':-1}

meta_x_model.set_params(params)

Instantiating the Meta model!

In [54]:
def compare(base: dict, meta: dict):
    for key in base:
        diff = meta[key] - base[key]
        symbol = "↑" if diff > 0 else ("↓" if diff < 0 else "=")
        print(f"{key}: base={base[key]:.4f} meta={meta[key]:.4f} {symbol} ({diff:+.4f})")

Function to compare the base models performance with the meta models performance!

In [56]:
meta_x_model.train(oof,y_train)
lr_proba=meta_x_model.predict_proba(oof_test)
preds=meta_x_model.predict(oof_test)


Training the meta model and getting the predictions and prediction probabilities !

In [57]:

def measure(y_pred: np.ndarray, y_proba: np.ndarray, y_real: np.ndarray) -> dict:
    metrics = {}
    metrics['accuracy'] = accuracy_score(y_real, y_pred)
    metrics['balanced']=balanced_accuracy_score(y_real,y_pred)
    metrics['precision'] = precision_score(y_real, y_pred)
    metrics['f1'] = f1_score(y_real, y_pred)
    metrics['recall'] = recall_score(y_real, y_pred)
    metrics['roc_auc'] = roc_auc_score(y_real, y_proba[:,1])
    return metrics

    

Function to measure meta models performance!

In [58]:
logistic_regression_metrics=measure(preds,lr_proba,y_test)
print(logistic_regression_metrics)

{'accuracy': 0.8304093567251462, 'balanced': 0.8301466596394733, 'precision': 0.7652370203160271, 'f1': 0.795774647887324, 'recall': 0.8288508557457213, 'roc_auc': 0.9275419749319405}


In [ ]:

metrics=meta_x_model.evaluation(oof,y_train,5)
metrics.out()

Accuracy: 0.8250054959745701, std: 0.004353747685608437
Precision: 0.7507557491652934, std: 0.010044182693977066
F1: 0.7825785829654129, std: 0.007458405256324119
Recall: 0.8178133610190471, std: 0.022133057409445208
Balanced Accuracy:0.8236552267927234, std: 0.006430523072245744
ROC-AUC: 0.9136381912894513, std: 0.0069368923281369005


Cross validating the meta model!

In [60]:
for model in ensemble:
    if model.__class__.__name__ in ["XGBoost", "RandomForest","NB","Ada"]:
        model.train(X_train,y_train)
        print(model.__class__.__name__)
    else:
        model.train(X_train_copy,y_train)

RandomForest
XGBoost
Ada


Training the base models!

In [61]:
for model in ensemble:
    print(f'Model:{model}')
    if model.__class__.__name__ in ["XGBoost", "RandomForest","NB","Ada"]:
        predictions=model.predict(X_test)
        proba=model.predict_proba(X_test)
        vals=measure(predictions,proba,y_test)
        compare(vals,logistic_regression_metrics)
    else:
        predictions=model.predict(X_test_copy)
        proba=model.predict_proba(X_test_copy)
        vals=measure(predictions,proba,y_test)
        compare(vals,logistic_regression_metrics)

Model:<modeling.RandomForest.RandomForest object at 0x0000022AB8FDE490>
accuracy: base=0.8275 meta=0.8304 ↑ (+0.0029)
balanced: base=0.8112 meta=0.8301 ↑ (+0.0189)
precision: base=0.8169 meta=0.7652 ↓ (-0.0517)
f1: base=0.7716 meta=0.7958 ↑ (+0.0242)
recall: base=0.7311 meta=0.8289 ↑ (+0.0978)
roc_auc: base=0.9270 meta=0.9275 ↑ (+0.0005)
Model:<modeling.XGBoost.XGBoost object at 0x0000022AB8FDED50>
accuracy: base=0.8275 meta=0.8304 ↑ (+0.0029)
balanced: base=0.8121 meta=0.8301 ↑ (+0.0181)
precision: base=0.8135 meta=0.7652 ↓ (-0.0483)
f1: base=0.7728 meta=0.7958 ↑ (+0.0230)
recall: base=0.7359 meta=0.8289 ↑ (+0.0929)
roc_auc: base=0.9237 meta=0.9275 ↑ (+0.0038)
Model:<modeling.Ada.Ada object at 0x0000022AB8FDF110>
accuracy: base=0.8265 meta=0.8304 ↑ (+0.0039)
balanced: base=0.8211 meta=0.8301 ↑ (+0.0090)
precision: base=0.7757 meta=0.7652 ↓ (-0.0104)
f1: base=0.7850 meta=0.7958 ↑ (+0.0108)
recall: base=0.7946 meta=0.8289 ↑ (+0.0342)
roc_auc: base=0.9156 meta=0.9275 ↑ (+0.0119)
Model:<m

Comparing Base models with Meta model performance, we can see that meta model beats every model on most of metrics, loses to few of them on precision and loses to MLP on accuracy.However its still better than the base class models so sticking with meta model! 

In [62]:
feature_cols=oof.columns
meta_x_model.feature_importance(feature_cols)
print(feature_cols)

            feature  coefficient
0  OOF_RandomForest     3.243522
3           OOF_MLP     1.823511
2           OOF_Ada     1.693686
1       OOF_XGBoost     1.034897
Index(['OOF_RandomForest', 'OOF_XGBoost', 'OOF_Ada', 'OOF_MLP'], dtype='str')


As we can see Random Forest seems to be most important feature in the oof dataset used by the meta model to make predictions!